In [1]:
from google.colab import drive
import os

drive.mount('/content/drive') #Mounting drive (my google drive)

# Main Project Root
PROJECT_PATH = '/content/drive/MyDrive/Face_Recognition_Project/'

# 2. Where the MTCNN cropped faces will go (The "Output" for Notebook 2)
PROCESSED_DATA_PATH = os.path.join(PROJECT_PATH, 'processed_faces/')


Mounted at /content/drive


In [2]:
import kagglehub
import os
from google.colab import drive

# 3. Download the PINS dataset
path = kagglehub.dataset_download("hereisburak/pins-face-recognition")

# 4. Locate the specific folder (it usually has a nested '105_classes_pins_dataset')
RAW_DATA_PATH = os.path.join(path, '105_classes_pins_dataset')

# Verify the path exists
if os.path.exists(RAW_DATA_PATH):
    print(f"✅ Success! Data found at: {RAW_DATA_PATH}")
    people = [d for d in os.listdir(RAW_DATA_PATH) if os.path.isdir(os.path.join(RAW_DATA_PATH, d))]
    print(f"✅ Found {len(people)} celebrities to recognize.")
else:
    print("❌ Path not found. Let's list the directory to find the correct folder name.")
    print(os.listdir(path))

100%|██████████| 372M/372M [00:03<00:00, 119MB/s]

Extracting files...


✅ Success! Data found at: /root/.cache/kagglehub/datasets/hereisburak/pins-face-recognition/versions/1/105_classes_pins_dataset
✅ Found 105 celebrities to recognize.


In [3]:
!pip install mtcnn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.4 MB/s eta 0:00:00


In [5]:
from mtcnn import MTCNN
import cv2
import os

# Initialize the MTCNN detector once to reuse across the loop
detector = MTCNN()

# Configuration constants for data quality control
MIN_FACE_SIZE = 48    # Minimum pixel width/height ensures faces aren't too small
MIN_CONFIDENCE = 0.90 # ensure we only save human faces

def process_pins_dataset(limit_per_person=20):
    """
    Crawls the PINS dataset, detects faces, crops them to 160x160,
    and saves them to the Google Drive 'Bridge'.
    """
    # Create a list of all folders (each representing a person) in the raw data path
    people_folders = [f for f in os.listdir(RAW_DATA_PATH) if os.path.isdir(os.path.join(RAW_DATA_PATH, f))]

    for person_folder in people_folders:
        # Data Cleaning: cleaning labels (names) by removing the werd 'pins_'
        clean_name = person_folder.replace('pins_', '')
        save_folder = os.path.join(PROCESSED_DATA_PATH, clean_name)

        # Create the destination folder in Drive if it doesn't already exist
        if not os.path.exists(save_folder):
            os.makedirs(save_folder)

        source_person_path = os.path.join(RAW_DATA_PATH, person_folder)
        # Select a subset of images per person to keep the dataset balanced and efficient
        image_files = os.listdir(source_person_path)[:limit_per_person]

        print(f"Processing: {clean_name}...", end=" ")

        for img_name in image_files:
            img_path = os.path.join(source_person_path, img_name)
            raw_img = cv2.imread(img_path)

            # Skip files that fail to load or are corrupted
            if raw_img is None:
                continue

            # Convert BGR (OpenCV default) to RGB (MTCNN/Matplotlib requirement)
            img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
            height, width, _ = img.shape

            # Preliminary check: skip images smaller than our minimum face requirement
            if height < MIN_FACE_SIZE or width < MIN_FACE_SIZE:
                continue

            # ✅ ERROR HANDLING: Wrap detection in try/except
            # to prevent crashes from internal MTCNN/TensorFlow engine errors
            try:
                results = detector.detect_faces(img)
            except Exception as e:
                print(f"\n  [Skipped] {img_name} — MTCNN error: {e}")
                continue

            if results:
                # MTCNN returns a list of faces; we assume the primary subject is index 0
                best = results[0]

                # Filter out low-confidence detections (e.g., patterns that look like faces)
                if best['confidence'] < MIN_CONFIDENCE:
                    continue

                # Get bounding box: x, y (top-left) and width, height
                x, y, w, h = best['box']

                # BOUNDARY LOGIC: Clip coordinates to [0, width/height]
                # to prevent negative index errors during cropping
                x1, y1 = max(0, x), max(0, y)
                x2, y2 = min(width, x + w), min(height, y + h)

                # Perform the crop using NumPy slicing
                face = img[y1:y2, x1:x2]

                # Final check: Ensure the resulting crop is valid and meets size requirements
                if face.size == 0 or face.shape[0] < MIN_FACE_SIZE or face.shape[1] < MIN_FACE_SIZE:
                    continue

                # STANDARDIZATION: Resize to 160x160 for the FaceNet model in Notebook 2
                face_resized = cv2.resize(face, (160, 160))

                # Save the final face back to Drive, converting back to BGR for imwrite
                cv2.imwrite(
                    os.path.join(save_folder, img_name),
                    cv2.cvtColor(face_resized, cv2.COLOR_RGB2BGR)
                )
        print("Done.")

# Execution call: limit images to 20 per class for a balanced training set
process_pins_dataset(limit_per_person=20)

print("\n--- Data processing is complete, the cropped faces are saved to drive---")

Processing: Adriana Lima... Done.
Processing: Selena Gomez... Done.
Processing: Marie Avgeropoulos... Done.
Processing: Jessica Barden... Done.
Processing: Jason Momoa... Done.
Processing: Jennifer Lawrence... Done.
Processing: melissa fumero... Done.
Processing: Penn Badgley... Done.
Processing: Zendaya... Done.
Processing: Emma Watson... Done.
Processing: Ursula Corbero... 
  [Skipped] Ursula Corbero10_2.jpg — MTCNN error: Exception encountered when calling Conv2D.call().

The convolution operation resulted in an empty output. Output shape: (0, 46, 46, 32). This can happen if the input is too small for the given kernel size, strides, dilation rate, and padding mode. Please check the input shape and convolution parameters.

Arguments received by Conv2D.call():
  • inputs=tf.Tensor(shape=(0, 48, 48, 3), dtype=float32)
Done.
Processing: amber heard... Done.
Processing: Avril Lavigne... Done.
Processing: Dominic Purcell... Done.
Processing: Rami Malek... Done.
Processing: Tuppence Middle